# 01 — First circuit and load flow

**Goal:** run a small source → line → load Case, inspect the solved network, then verify the saved evidence.

**Teaching inputs:** 12.47 kV three-phase system, 1.0 km line, 100 kW load at PF 0.95. These are demonstrator values, not measurements.

**Prediction:** the load-bus voltage should be slightly below the 1.0 pu source voltage.

Run the numbered cells in order. The direct OpenDSS section at the end is optional.

In [1]:
#@title 1. Setup — run once
from hashlib import sha256
from urllib.request import urlopen

_bootstrap_url = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/notebooks/_lesson.py"
_bootstrap = urlopen(_bootstrap_url, timeout=60).read()
if sha256(_bootstrap).hexdigest() != "fd7df585358a9e127d0976f9a804e3c4cb30ce5c07c5ff1b0c38e3af19233d72":
    raise ValueError("Lesson helper hash mismatch")
exec(compile(_bootstrap, "cept-lesson", "exec"), globals())


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [2]:
#@title 2. Inputs — source, line, and load
CASE_PATH = WORKSPACE / "first_circuit_case.json"
CASE_PAYLOAD = first_circuit_case()
CASE_PATH.write_text(json.dumps(CASE_PAYLOAD, indent=2) + "\n", encoding="utf-8")
RUN_DIR = WORKSPACE / "runs" / "01-first-circuit"

table(
    ["declared input", "value", "unit"],
    [
        ("frequency", 60, "Hz"),
        ("line length", 1.0, "km"),
        ("load", 100.0, "kW"),
        ("load power factor", 0.95, "1"),
    ],
)

declared input     value  unit
-----------------  -----  ----
frequency          60     Hz
line length        1.0    km
load               100.0  kW
load power factor  0.95   1


In [3]:
# 3. Run study — same CEPT CLI as a normal terminal
!cept study run first_circuit_case.json \
    --out runs/01-first-circuit \
    --force \
    --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the study and saved the evidence
Saved run          runs\01-first-circuit
Case fingerprint   aaea341ddd26 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\01-first-circuit --format text


In [4]:
#@title 4. Explore — SLD and bus status
RUN_DIR = WORKSPACE / "runs" / "01-first-circuit"
# Use the local compatibility renderer so older public wheels cannot hide SLD edges.
display_run = display_run_compat
display_run(RUN_DIR)

Bus,Phase A,Phase B,Phase C,Status
source,1.0000 pu @ 0.00°,1.0000 pu @ -120.00°,1.0000 pu @ 120.00°,OK
load,0.9998 pu @ -0.01°,0.9998 pu @ -120.01°,0.9998 pu @ 119.99°,OK
grid,1.0000 pu @ 0.00°,1.0000 pu @ -120.00°,1.0000 pu @ 120.00°,OK


In [5]:
#@title 5. Engineering result — persisted solver values
results = read(RUN_DIR / "results.json")
load_flow = results["load_flow"]
table(
    ["quantity", "value", "unit"],
    [
        ("converged", load_flow["converged"], "bool"),
        ("total load", load_flow["total_load_kw"], "kW"),
        ("total loss", load_flow["total_loss_kw"], "kW"),
        ("source P", load_flow["source_p_kw"], "kW"),
    ],
)
assert load_flow["converged"] is True

quantity    value     unit
----------  --------  ----
converged   True      bool
total load  100.0     kW
total loss  0.0143    kW
source P    100.0141  kW


In [ ]:
#@title 6. Plot — source vs load voltage and power balance
import matplotlib.pyplot as plt

lf = results["load_flow"]
load_rows = [r for r in lf["bus_voltages"] if r["bus"].lower() == "load"]
source_rows = [r for r in lf["bus_voltages"] if r["bus"].lower() == "source"]
v_load = [next(float(r["v_pu"]) for r in load_rows if r["phase"] == p) for p in (1, 2, 3)]
v_src = [next(float(r["v_pu"]) for r in source_rows if r["phase"] == p) for p in (1, 2, 3)]
drop_pct = [(s - l) / s * 100.0 for s, l in zip(v_src, v_load)]
loss_kw = float(lf.get("total_loss_kw") or 0.0)
load_kw = float(lf.get("total_load_kw") or 0.0)
src_kw = float(lf.get("source_p_kw") or (load_kw + loss_kw))

fig, (axV, axP) = plt.subplots(1, 2, figsize=(10, 4))
x = [0, 1, 2]; w = 0.35
axV.bar([v - w / 2 for v in x], v_src, w, label="Source", color="#9db8ad", edgecolor="white")
axV.bar([v + w / 2 for v in x], v_load, w, label="Load", color="#1f5b4d", edgecolor="white")
axV.set_xticks(x); axV.set_xticklabels(["A", "B", "C"])
axV.set_title("Voltage: source vs load (zoomed)")
axV.set_xlabel("Phase"); axV.set_ylabel("Voltage (pu)")
vmin, vmax = min(v_src + v_load), max(v_src + v_load)
pad = max(0.0006, (vmax - vmin) * 1.6)
axV.set_ylim(vmin - pad, vmax + pad)
axV.legend(frameon=False, fontsize=8); axV.grid(axis="y", alpha=0.25)
for i, d in enumerate(drop_pct):
    axV.text(i, v_load[i] - pad * 0.12, f"drop {d:.3f}%", ha="center", va="top", fontsize=8, color="#334155")

axP.bar([0], [load_kw], width=0.5, label=f"Load {load_kw:.1f} kW", color="#1f5b4d")
axP.bar([0], [loss_kw], width=0.5, bottom=[load_kw], label=f"Loss {loss_kw:.4f} kW", color="#d5654e")
axP.set_xticks([0]); axP.set_xticklabels(["Source \u2192 load + loss"])
axP.set_title(f"Power balance (source \u2248 {src_kw:.2f} kW)")
axP.set_ylabel("Active power (kW)"); axP.set_ylim(0, max(src_kw, load_kw + loss_kw) * 1.25 or 1.0)
axP.legend(frameon=False, fontsize=8); axP.grid(axis="y", alpha=0.25)
loss_share = (loss_kw / src_kw * 100.0) if src_kw else 0.0
axP.text(0, load_kw + loss_kw * 0.5, f"loss {loss_share:.3f}%", ha="center", va="center", fontsize=8, color="white", weight="bold")

fig.suptitle("First circuit: stiff source, tiny drop — as predicted", fontsize=11)
fig.tight_layout()
from IPython.display import display
display(fig)
plt.close(fig)


In [7]:
# 6. Verify — check this exact saved run
!cept study verify runs/01-first-circuit --format text

CEPT study check: PASSED
----------------------------
Study              Load flow (OpenDSS)
Case fingerprint   aaea341ddd26 (matches the case you ran)

Checked   3 groups, 14 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (3 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     public-verification.json

For the full check list
  cept study verify . --format json


## 7. Interpret

The load flow converged and the SLD/bus table show the solver-backed operating point for this declared demonstrator.

**What this proves:** this CEPT workflow ran, saved its result, and can be verified against its Case and artifacts.

**What this does not prove:** acceptance of a real feeder or field installation.

**Try next:** change one declared input in CASE_PAYLOAD, predict the direction of change, restart the kernel, and rerun the lesson.

## Optional — direct OpenDSS comparison

The cells below are for learners who want to inspect the solver route. They are not required for the main CEPT workflow. Run them in order to compare direct OpenDSS readback with the persisted CEPT result.

In [8]:
#@title Under the hood - direct OpenDSS solve (optional)
import opendssdirect as dss

for command in [
    'Clear',
    'New Circuit.first basekv=12.47 pu=1.0 phases=3 bus1=source',
    'New Line.line1 bus1=source.1.2.3 bus2=load.1.2.3 phases=3 length=1 units=km r1=0.2 x1=0.4 r0=0.6 x0=1.2 c1=0 c0=0',
    'New Load.load1 bus1=load.1.2.3 phases=3 conn=wye kv=12.47 kw=100 pf=0.95',
    'CalcVoltageBases',
    'Solve',
]:
    dss.Text.Command(command)
assert dss.Solution.Converged()
print("Direct OpenDSS solve finished: the circuit converged.")


Direct OpenDSS solve finished: the circuit converged.


In [9]:
#@title Under the hood — direct OpenDSS readback (optional)
dss.Circuit.SetActiveBus('load')
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}
table(['source', 'phase', 'voltage magnitude', 'unit'], [('direct OpenDSS', phase, direct_by_phase[phase], 'pu') for phase in (1, 2, 3)])
assert all(value > 0 for value in direct_by_phase.values())

source          phase  voltage magnitude   unit
--------------  -----  ------------------  ----
direct OpenDSS  1      0.9997586726902581  pu
direct OpenDSS  2      0.9997586726902756  pu
direct OpenDSS  3      0.9997586726902107  pu


In [10]:
#@title Compare solver outputs (optional details)
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
cept_rows = [row for row in results["load_flow"]["bus_voltages"] if row["bus"].lower() == "load"]
cept_by_phase = {row["phase"]: row["v_pu"] for row in cept_rows}
table(
    ["phase", "direct OpenDSS pu", "CEPT pu", "|difference| pu"],
    [(phase, direct_by_phase[phase], cept_by_phase[phase], abs(direct_by_phase[phase] - cept_by_phase[phase])) for phase in (1, 2, 3)],
)
max_abs_diff_pu = max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3))
print()
print("Direct OpenDSS and CEPT agree")
print("-----------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Largest phase difference: {max_abs_diff_pu:.2e} per unit (limit 1e-4 per unit)")
print("A small difference is rounding, not a different answer.")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert set(cept_by_phase) == {1, 2, 3}
assert max_abs_diff_pu < 1e-4


phase  direct OpenDSS pu   CEPT pu   |difference| pu
-----  ------------------  --------  ----------------------
1      0.9997586726902581  0.999787  2.8327309741893458e-05
2      0.9997586726902756  0.999787  2.8327309724351935e-05
3      0.9997586726902107  0.999787  2.832730978929998e-05

Direct OpenDSS and CEPT agree
-----------------------------
Result        PASSED
Largest phase difference: 2.83e-05 per unit (limit 1e-4 per unit)
A small difference is rounding, not a different answer.
